# Prediction notebook (platform version)

Upload **this notebook** (card 1) together with **`model.pkl`** (card 2).
The platform runs it top to bottom in a sandbox with no internet and no `train.csv`, so this notebook only:

1. loads `model.pkl` (every model was already trained on my machine with the platform's library versions),
2. builds the same features as in training,
3. writes one prediction per row of the test file, in the same row order, under the column `prediction`.

`model.pkl` holds two versions; pick one with `VERSION` below:

| VERSION | what it is | public score |
|---|---|---|
| `"final"` | 0.30 × v4 + 0.70 × LightGBM-imputed model, averaged over 10 random seeds (steadier, recommended) | about 3.28 |
| `"best"` | 0.25 × v4 + 0.75 × LightGBM-imputed model, seed 527 only | about 3.27 |

Only standard libraries from `requirements-image.txt` are used (numpy, pandas, scikit-learn, lightgbm, joblib).

In [ ]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")

VERSION = "final"          # "final" (10 seeds, recommended) or "best" (seed 527)

## Feature code
Copied from the training code. The rows are never matched by id or position, so a test file with no `id` column and shuffled rows works.

In [ ]:
NUMERIC_COLUMNS = ['temperature', 'humidity', 'occupancy', 'previous_usage']
BUILDING_IDS = ['ADM_A', 'BUS_A', 'BUS_B', 'ENG_A', 'ENG_B', 'LEC_A', 'LIB_A', 'RES_A', 'RES_B', 'SCI_A', 'SCI_B', 'SPT_A']
BUILDING_TYPES = ['Administration', 'Business', 'Engineering', 'LectureHall', 'Library', 'Residential', 'Science', 'Sports']
DAY_NAMES = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
DAY_NUMBER = {day: number for number, day in enumerate(DAY_NAMES)}
BUILDING_NUMBER = {building: number for number, building in enumerate(BUILDING_IDS)}
IMPUTER_CATEGORIES = {'building_id': BUILDING_IDS, 'building_type': BUILDING_TYPES, 'day_of_week': DAY_NAMES}


def is_weekend(rows):
    return rows['day_of_week'].isin(['Saturday', 'Sunday'])

### Features shared by the models

In [ ]:
# ---- features shared by the v3 model and the LightGBM-imputed model ----
def missforest_inputs(rows):
    inputs = rows[NUMERIC_COLUMNS].copy()
    inputs['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    inputs['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    inputs['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    inputs['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    inputs['is_weekend'] = is_weekend(rows).astype(float)
    building_onehot = pd.get_dummies(pd.Categorical(rows['building_id'], categories=BUILDING_IDS), prefix='bid').astype(float)
    building_onehot.index = rows.index
    return pd.concat([inputs, building_onehot], axis=1)


def ridge_features(rows, typical):
    features = rows[NUMERIC_COLUMNS].copy()
    features['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    features['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    features['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    features['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    features['is_weekend'] = is_weekend(rows).astype(float)
    features['occupancy_x_weekend'] = features['occupancy'] * features['is_weekend']
    building_onehot = pd.get_dummies(pd.Categorical(rows['building_id'], categories=BUILDING_IDS), prefix='bid').astype(float)
    type_onehot = pd.get_dummies(pd.Categorical(rows['building_type'], categories=BUILDING_TYPES), prefix='bt').astype(float)
    building_onehot.index = type_onehot.index = rows.index
    interactions = {}
    for column in ['occupancy', 'previous_usage', 'temperature', 'hour_sin', 'hour_cos']:
        for building in building_onehot.columns:
            interactions[f'{column}_x_{building}'] = features[column] * building_onehot[building]
    for column in ['is_weekend', 'occupancy_x_weekend']:
        for building_type in type_onehot.columns:
            interactions[f'{column}_x_{building_type}'] = features[column] * type_onehot[building_type]
    lookup_keys = pd.DataFrame({'building_id': rows['building_id'], 'hour': rows['hour'], 'is_weekend': features['is_weekend'] > 0})
    typical_for_row = lookup_keys.join(typical, on=['building_id', 'hour', 'is_weekend']).fillna(typical.median())
    events = pd.DataFrame({
        'occupancy_gap': features['occupancy'] - typical_for_row['occupancy'],
        'previous_ratio': features['previous_usage'] / typical_for_row['previous_usage'],
        'previous_gap': features['previous_usage'] - typical_for_row['previous_usage'],
        'heat': (features['temperature'] >= 33.5).astype(float),
        'rain': (features['humidity'] >= 93).astype(float),
    }, index=rows.index)
    for column in events.columns:
        for building_type in type_onehot.columns:
            interactions[f'{column}_x_{building_type}'] = events[column] * type_onehot[building_type]
    part_of_day = pd.cut(rows['hour'], [-1, 5, 8, 11, 14, 17, 20, 23], labels=False)
    part_onehot = pd.get_dummies(pd.Categorical(part_of_day, categories=range(7)), prefix='dp').astype(float)
    part_onehot.index = rows.index
    for part in part_onehot.columns:
        for building_type in type_onehot.columns:
            interactions[f'occupancy_{part}_x_{building_type}'] = features['occupancy'] * part_onehot[part] * type_onehot[building_type]
    return pd.concat([features, type_onehot, building_onehot, events, pd.DataFrame(interactions, index=rows.index)], axis=1)


def tree_features(rows):
    features = rows[['hour', 'month'] + NUMERIC_COLUMNS].copy()
    features['day'] = rows['day_of_week'].map(DAY_NUMBER)
    features['is_weekend'] = (features['day'] >= 5).astype(int)
    features['building_type'] = pd.Categorical(rows['building_type'], categories=BUILDING_TYPES)
    features['building_id'] = pd.Categorical(rows['building_id'], categories=BUILDING_IDS)
    return features

### v4, part 1: the v3 notebook model

In [ ]:
# ---- v4 part 1: the v3 notebook model (MissForest filling, Ridge 80% + boosting 20%) ----
def predict_v3(parts, rows):
    filled = rows.copy()
    filled[NUMERIC_COLUMNS] = parts['imputer'].transform(missforest_inputs(rows))[:, :len(NUMERIC_COLUMNS)]
    ridge_prediction = parts['ridge'].predict(ridge_features(filled, parts['typical']))
    boosting_prediction = parts['boosting'].predict(tree_features(filled)) + filled['previous_usage'].to_numpy()
    return 0.8 * ridge_prediction + 0.2 * boosting_prediction

### v4, part 2: one model per pattern of missing inputs

In [ ]:
# ---- v4 part 2: one model per pattern of missing inputs ----
def pattern_model_inputs(rows, available_columns, n_daily_curves=3):
    hour = rows['hour'].to_numpy(float)
    columns = {name: rows[name].to_numpy(float) for name in available_columns}
    columns['weekend'] = is_weekend(rows).to_numpy(float)
    for k in range(1, n_daily_curves + 1):
        columns[f'hour_sin_{k}'] = np.sin(2 * np.pi * k * hour / 24)
        columns[f'hour_cos_{k}'] = np.cos(2 * np.pi * k * hour / 24)
    columns['month_sin'] = np.sin(2 * np.pi * rows['month'].to_numpy() / 12)
    columns['month_cos'] = np.cos(2 * np.pi * rows['month'].to_numpy() / 12)
    return np.column_stack(list(columns.values()))


def pattern_ridge_inputs(rows, available_columns, scaler):
    scaled = scaler.transform(pattern_model_inputs(rows, available_columns))
    building_onehot = (rows['building_id'].to_numpy()[:, None] == np.array(BUILDING_IDS)[None, :]).astype(float)
    per_building = (scaled[:, :, None] * building_onehot[:, None, :]).reshape(len(scaled), -1)
    return np.column_stack([scaled, building_onehot, per_building])


def pattern_correction_inputs(rows, available_columns):
    numeric = rows[['hour', 'month'] + available_columns].to_numpy(float)
    return np.column_stack([numeric, rows['day_of_week'].map(DAY_NUMBER), rows['building_id'].map(BUILDING_NUMBER)])


def predict_patterns(pattern_models, rows, correction_weight=0.65):
    blanks = rows[NUMERIC_COLUMNS].isna().to_numpy().astype(int)
    patterns = (blanks * 2 ** np.arange(len(NUMERIC_COLUMNS))).sum(axis=1)
    prediction = np.empty(len(rows))
    for pattern in np.unique(patterns):
        positions = np.flatnonzero(patterns == pattern)
        pattern_rows = rows.iloc[positions]
        available, scaler, ridge, correction = pattern_models[pattern]
        prediction[positions] = (ridge.predict(pattern_ridge_inputs(pattern_rows, available, scaler))
                                 + correction_weight * correction.predict(pattern_correction_inputs(pattern_rows, available)))
    return prediction


def predict_v4(model, rows):
    return 0.25 * predict_v3(model['v3'], rows) + 0.75 * predict_patterns(model['pattern_models'], rows)

### The LightGBM-imputed model

In [ ]:
# ---- the LightGBM-imputed model ----
def imputer_inputs(rows):
    inputs = rows[['building_id', 'building_type', 'hour', 'day_of_week', 'month'] + NUMERIC_COLUMNS].copy()
    inputs['weekend'] = is_weekend(rows).astype(int)
    inputs['hour_sin'] = np.sin(2 * np.pi * rows['hour'] / 24)
    inputs['hour_cos'] = np.cos(2 * np.pi * rows['hour'] / 24)
    inputs['month_sin'] = np.sin(2 * np.pi * rows['month'] / 12)
    inputs['month_cos'] = np.cos(2 * np.pi * rows['month'] / 12)
    for column, categories in IMPUTER_CATEGORIES.items():
        inputs[column] = pd.Categorical(inputs[column], categories=categories)
    return inputs


def predict_lightgbm_imputed(parts, rows):
    imputers = parts['imputers']
    expected = lambda frame, column: imputers[column].predict(imputer_inputs(frame).drop(columns=column))
    filled = rows.copy()
    for column in NUMERIC_COLUMNS:                       # fill each blank with the imputer's expected value
        blank = rows[column].isna()
        if blank.any():
            filled.loc[blank, column] = expected(rows.loc[blank], column)
    features = ridge_features(filled, parts['typical']).copy()
    expected_previous, expected_occupancy = expected(rows, 'previous_usage'), expected(rows, 'occupancy')
    features['expected_previous'] = expected_previous
    features['expected_occupancy'] = expected_occupancy
    for building_type in BUILDING_TYPES:
        this_type = (filled['building_type'] == building_type).to_numpy(float)
        features[f'expected_previous_{building_type}'] = expected_previous * this_type
        features[f'expected_occupancy_{building_type}'] = expected_occupancy * this_type
    ridge_prediction = parts['ridge'].predict(features)
    boosting_prediction = parts['boosting'].predict(tree_features(filled)) + filled['previous_usage'].to_numpy()
    return 0.8 * ridge_prediction + 0.2 * boosting_prediction


def predict_final(model, rows, seeds, weight_on_lightgbm_model):
    v4 = predict_v4(model, rows)
    lightgbm_model = np.mean([predict_lightgbm_imputed(model['lightgbm_imputed'][seed], rows) for seed in seeds], axis=0)
    return (1 - weight_on_lightgbm_model) * v4 + weight_on_lightgbm_model * lightgbm_model

## Load the trained models and predict

In [ ]:
model = joblib.load("model.pkl")          # uploaded in card 2, next to this notebook
settings = model["versions"][VERSION]

test_rows = pd.read_csv(TEST_PATH)
test_rows = test_rows.drop(columns=["id", "energy_usage"], errors="ignore")

preds = predict_final(model, test_rows, settings["seeds"], settings["weight_on_lightgbm_model"])
print(f"version {VERSION!r}: {len(settings['seeds'])} seed(s), weight on LightGBM-imputed model {settings['weight_on_lightgbm_model']}")
print(f"{len(test_rows):,} rows | blank values: {int(test_rows.isna().sum().sum())} | predictions {preds.min():.1f} to {preds.max():.1f}")
assert len(preds) == len(test_rows) and np.isfinite(preds).all()

In [ ]:
pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")